In [2]:
import gzip
import io
import json
import re
import requests
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from langchain_core.documents import Document

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

In [3]:
# Config
CATEGORY       = "All_Beauty"          # Change to any category from the dataset
N_SUBSET       = 200                   # Number of records to load for EDA
BASE_URL       = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
REVIEW_URL     = f"{BASE_URL}/review_categories/{CATEGORY}.jsonl.gz"
META_URL       = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"

print(f"Category : {CATEGORY}")
print(f"Review URL: {REVIEW_URL}")
print(f"Meta URL  : {META_URL}")

Category : All_Beauty
Review URL: https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/All_Beauty.jsonl.gz
Meta URL  : https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_All_Beauty.jsonl.gz


In [4]:
def stream_jsonl_gz(url: str, n: int = 200) -> list[dict]:
    """
    Stream the first `n` records from a remote .jsonl.gz file.
    Downloads only as much data as needed.
    """
    print(f"Fetching up to {n} records from:\n  {url}")
    records = []
    with requests.get(url, stream=True, timeout=60) as resp:
        resp.raise_for_status()
        # Decompress the stream on the fly
        with gzip.open(io.BytesIO(resp.content), "rt", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= n:
                    break
                records.append(json.loads(line.strip()))
    print(f"  → Loaded {len(records)} records.")
    return records


reviews_raw = stream_jsonl_gz(REVIEW_URL, n=N_SUBSET)
meta_raw    = stream_jsonl_gz(META_URL,   n=N_SUBSET)

Fetching up to 200 records from:
  https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/All_Beauty.jsonl.gz
  → Loaded 200 records.
Fetching up to 200 records from:
  https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_All_Beauty.jsonl.gz
  → Loaded 200 records.


In [7]:
reviews_raw[:5]

[{'rating': 5.0,
  'title': 'Such a lovely scent but not overpowering.',
  'text': "This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!",
  'images': [],
  'asin': 'B00YQ6X8EO',
  'parent_asin': 'B00YQ6X8EO',
  'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ',
  'timestamp': 1588687728923,
  'helpful_vote': 0,
  'verified_purchase': True},
 {'rating': 4.0,
  'title': 'Works great but smells a little weird.',
  'text': 'This product does what I need it to do, I just wish it was odorless or had a soft coconut smell. Having my head smell like an orange coffee is offputting. (granted, I did know the smell was described but I was hoping it would be light)',
  'images': [],
  'asin': 'B081TJ8YS3',
  'parent_asin': 'B081TJ8YS3',
  'user_id': 'AGKHLEW2SOWH

In [9]:
meta_raw[:5]

[{'main_category': 'All Beauty',
  'title': 'Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)',
  'average_rating': 4.8,
  'rating_number': 10,
  'features': [],
  'description': [],
  'price': None,
  'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qfjSfqNyL._SS40_.jpg',
    'large': 'https://m.media-amazon.com/images/I/41qfjSfqNyL.jpg',
    'variant': 'MAIN',
    'hi_res': None},
   {'thumb': 'https://m.media-amazon.com/images/I/41w2yznfuZL._SS40_.jpg',
    'large': 'https://m.media-amazon.com/images/I/41w2yznfuZL.jpg',
    'variant': 'PT01',
    'hi_res': 'https://m.media-amazon.com/images/I/71i77AuI9xL._SL1500_.jpg'}],
  'videos': [],
  'store': 'Howard Products',
  'categories': [],
  'details': {'Package Dimensions': '7.1 x 5.5 x 3 inches; 2.38 Pounds',
   'UPC': '617390882781'},
  'parent_asin': 'B01CUPMQZE',
  'bought_together': None},
 {'main_category': 'All Beauty',
  'title': 'Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Toma

In [ ]:
reviews_df = pd.DataFrame(reviews_raw)
meta_df    = pd.DataFrame(meta_raw)

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588687728923,0,True
1,4.0,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588615855070,1,True
2,5.0,Yes!,"Smells good, feels great!",[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,1589665266052,2,True
3,1.0,Synthetic feeling,Felt synthetic,[],B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,1643393630220,0,True
4,5.0,A+,Love it,[],B08BZ63GMJ,B08BZ63GMJ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,1609322563534,0,True


In [ ]:
# load user reviews
file = "Musical_Instruments.jsonl"
with open(file, 'r') as fp:
    for line in fp:
        print(json.loads(line.strip()))


FileNotFoundError: [Errno 2] No such file or directory: 'Musical_Instruments.jsonl'

In [ ]:
# load metadata

file = # e.g., "meta_All_Beauty.jsonl", downloaded from the `meta` link above
with open(file, 'r') as fp:
    for line in fp:
        print(json.loads(line.strip()))
